# [16.1] Exact Shapley on Ground-Truth Games - Solutions

Reference validation notebook for the section-local exact Shapley implementation and committed CUDA neural-game verification report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part1_exact_shapley_ground_truth_games"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_exact_shapley_ground_truth_games.tests as tests
from chapter16_shapley_attribution_baselines.exercises.part1_exact_shapley_ground_truth_games import solutions

In [ ]:
tests.test_all_coalitions_enumerates_the_power_set(solutions.all_coalitions)
tests.test_additive_game_and_exact_shapley_recover_weights(
    solutions.additive_game,
    solutions.exact_shapley_values,
)
tests.test_exact_shapley_requires_a_complete_coalition_table(
    solutions.exact_shapley_values,
)
tests.test_conjunction_game_splits_symmetric_credit_and_checks_efficiency(
    solutions.conjunction_game,
    solutions.exact_shapley_values,
    solutions.shapley_efficiency_report,
)
tests.test_permutation_parity_report_matches_exact_formula(
    solutions.conjunction_game,
    solutions.permutation_parity_report,
)
tests.test_interaction_gap_report_catches_leave_one_out_overcount(
    solutions.conjunction_game,
    solutions.interaction_gap_report,
)
tests.test_additive_smoke_test(solutions.additive_smoke_test)
tests.test_conjunction_smoke_test(solutions.conjunction_smoke_test)
tests.test_permutation_parity_smoke_test(solutions.permutation_parity_smoke_test)
tests.test_interaction_failure_smoke_test(solutions.interaction_failure_smoke_test)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["additive"]["shapley"] == [1.0, 2.0, -0.5], "Additive exact Shapley should recover the feature weights."
assert contract["additive"]["efficiency"]["satisfies_efficiency"], "Additive Shapley should satisfy efficiency."
assert contract["conjunction"]["shapley"] == [1 / 3, 1 / 3, 1 / 3], "Conjunction Shapley should split credit equally."
assert contract["conjunction"]["efficiency"]["satisfies_efficiency"], "Conjunction Shapley should satisfy efficiency."
assert contract["permutation_parity"]["matches_exact"], "Permutation averaging should match exact Shapley."
assert contract["interaction_failure"]["detects_interaction_overcount"], "Interaction game should reveal leave-one-out overcounting."
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"], "The committed verification report should be accepted."
assert gpu["cuda_available"], "The 16.1 release report should be a CUDA report."
assert gpu["preflight_passed"], "The neural exact-Shapley preflight should pass."
assert gpu["model_family"] == "cuda_trained_neural_coalition_game_mlp", "Unexpected neural-game model family."
assert gpu["num_players"] == 4, "The CUDA neural game should use four players."
assert gpu["coalition_count"] == 16, "A four-player exact game should have 16 coalitions."
assert gpu["training_example_count"] == 16, "The model should train on the complete binary feature table."
assert gpu["training_steps"] == 1200, "The pinned neural-game preflight should use 1200 training steps."
assert gpu["fit_mse"] <= 1e-8, "The neural game should fit the complete finite table."
assert gpu["fit_max_abs_error"] <= 1e-4, "The trained neural game should have tiny max fit error."
assert gpu["neural_shapley_max_abs_error"] <= 1e-4, "Model-ablation Shapley should match analytic Shapley."
assert gpu["satisfies_efficiency"], "Model-ablation Shapley should satisfy efficiency."
assert gpu["shuffled_control_error"] >= 1.0, "The shuffled-label control should be far from true Shapley."
assert gpu["shuffled_control_cosine"] <= 0.25, "The shuffled-label control should not align with true Shapley."
assert gpu["shuffled_control_rejected"], "The shuffled-label trained-model control should be rejected."
assert gpu["within_vram_budget"], "The report should stay within the configured VRAM budget."

{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "fit_mse",
    "neural_shapley_max_abs_error",
    "shuffled_control_error",
    "shuffled_control_cosine",
    "peak_vram_gb",
]}